<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/Optimize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving population.csv to population.csv


In [2]:
import os
import shutil
shutil.move('population.csv', '/content/population.csv')

'/content/population.csv'

In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("optimizeExample").getOrCreate()

In [4]:
df=spark.read.csv("/content/population.csv",header=True,inferSchema=True)

In [5]:
df.show(5)

+------------+------------+----+-------+
|Country Name|Country Code|Year|  Value|
+------------+------------+----+-------+
|       Aruba|         ABW|1960|54922.0|
|       Aruba|         ABW|1961|55578.0|
|       Aruba|         ABW|1962|56320.0|
|       Aruba|         ABW|1963|57002.0|
|       Aruba|         ABW|1964|57619.0|
+------------+------------+----+-------+
only showing top 5 rows



In [6]:
df.groupBy("Year").count().show()

+----+-----+
|Year|count|
+----+-----+
|1990|  265|
|1975|  264|
|1977|  264|
|2003|  265|
|2007|  265|
|2018|  265|
|1974|  264|
|2015|  265|
|2023|  265|
|2006|  265|
|1978|  264|
|2022|  265|
|1961|  264|
|2013|  265|
|1988|  264|
|1997|  265|
|1994|  265|
|1968|  264|
|2014|  265|
|1973|  264|
+----+-----+
only showing top 20 rows



In [18]:
df.select("Year").distinct().count()

64

In [7]:
from pyspark.sql.functions import avg

# Corrected code with unique aliases
df.groupBy("Country Name").agg(
    avg("Value").alias("Average Value")
).show()


+--------------------+-------------------+
|        Country Name|      Average Value|
+--------------------+-------------------+
|          South Asia|  1.2133345478125E9|
|                Chad|       8040221.1875|
| Lower middle income| 1.83420376521875E9|
|            Paraguay|     4215777.890625|
| Low & middle income|4.258080837546875E9|
|Heavily indebted ...|     4.3425936975E8|
|               World|  5.4587022159375E9|
|    Congo, Dem. Rep.|  4.5710188265625E7|
|             Senegal|     8870310.578125|
|          Cabo Verde|        391508.9375|
|              Sweden|     8753814.609375|
|East Asia & Pacif...|   1.567344636625E9|
|            Kiribati|        81998.15625|
|Least developed c...|  5.9721925196875E8|
|              Guyana|        741505.5625|
|             Eritrea|     2083260.234375|
|         Philippines|  6.7997814796875E7|
|Pacific island sm...|      1792425.40625|
|            Djibouti|        573017.6875|
|               Tonga|       96705.234375|
+----------

In [8]:
# Choose a valid column for analysis
valid_column = "saving"  # Change to any suitable column from your dataset

In [9]:
# Import the required module
import time

In [10]:
# Measure execution time with caching
df.cache()  # Cache the DataFrame
start_time_cached = time.time()
df.groupBy("Year").count().show()
end_time_cached = time.time()
print("Execution Time (Without Cache):", end_time_cached - start_time_cached)

+----+-----+
|Year|count|
+----+-----+
|1990|  265|
|1975|  264|
|1977|  264|
|2003|  265|
|2007|  265|
|2018|  265|
|1974|  264|
|2015|  265|
|2023|  265|
|2006|  265|
|1978|  264|
|2022|  265|
|1961|  264|
|2013|  265|
|1988|  264|
|1997|  265|
|1994|  265|
|1968|  264|
|2014|  265|
|1973|  264|
+----+-----+
only showing top 20 rows

Execution Time (Without Cache): 2.088704824447632


In [11]:
# Measure execution time with caching

start_time_cached = time.time()
df.groupBy("Year").count().show()
end_time_cached = time.time()
print("Execution Time (Without Cache):", end_time_cached - start_time_cached)

+----+-----+
|Year|count|
+----+-----+
|1990|  265|
|1975|  264|
|1977|  264|
|2003|  265|
|2007|  265|
|2018|  265|
|1974|  264|
|2015|  265|
|2023|  265|
|2006|  265|
|1978|  264|
|2022|  265|
|1961|  264|
|2013|  265|
|1988|  264|
|1997|  265|
|1994|  265|
|1968|  264|
|2014|  265|
|1973|  264|
+----+-----+
only showing top 20 rows

Execution Time (Without Cache): 0.6173570156097412


In [12]:

from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)

start_time = time.time()
df.groupBy("Country Name").count().show()
end_time = time.time()
print("Execution Time (With Persist):", end_time - start_time)

df.unpersist()

+--------------------+-----+
|        Country Name|count|
+--------------------+-----+
|          South Asia|   64|
|                Chad|   64|
| Lower middle income|   64|
|            Paraguay|   64|
| Low & middle income|   64|
|Heavily indebted ...|   64|
|               World|   64|
|    Congo, Dem. Rep.|   64|
|             Senegal|   64|
|          Cabo Verde|   64|
|              Sweden|   64|
|East Asia & Pacif...|   64|
|            Kiribati|   64|
|Least developed c...|   64|
|              Guyana|   64|
|             Eritrea|   64|
|         Philippines|   64|
|Pacific island sm...|   64|
|            Djibouti|   64|
|               Tonga|   64|
+--------------------+-----+
only showing top 20 rows

Execution Time (With Persist): 0.7791531085968018


DataFrame[Country Name: string, Country Code: string, Year: int, Value: double]

In [15]:
row_count = df.select("Year").distinct().count()
print("Number of unique years:", row_count)


Number of unique years: 64


In [19]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [20]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    " Value title",
    when(col("Value") >=55000, "High").otherwise("Low")
)
df.show()

+------------+------------+----+-------+------------+
|Country Name|Country Code|Year|  Value| Value title|
+------------+------------+----+-------+------------+
|       Aruba|         ABW|1960|54922.0|         Low|
|       Aruba|         ABW|1961|55578.0|        High|
|       Aruba|         ABW|1962|56320.0|        High|
|       Aruba|         ABW|1963|57002.0|        High|
|       Aruba|         ABW|1964|57619.0|        High|
|       Aruba|         ABW|1965|58190.0|        High|
|       Aruba|         ABW|1966|58694.0|        High|
|       Aruba|         ABW|1967|58990.0|        High|
|       Aruba|         ABW|1968|59069.0|        High|
|       Aruba|         ABW|1969|59052.0|        High|
|       Aruba|         ABW|1970|58950.0|        High|
|       Aruba|         ABW|1971|58781.0|        High|
|       Aruba|         ABW|1972|58047.0|        High|
|       Aruba|         ABW|1973|58299.0|        High|
|       Aruba|         ABW|1974|58349.0|        High|
|       Aruba|         ABW|1